# Extract Network Structure

In [ ]:
import pandas as pd
import sqlite3
import joblib
import os
from tqdm.notebook import tqdm
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from torch.utils.data import TensorDataset
import sys
from torch.utils.data import ConcatDataset, DataLoader

# set working directory
os.chdir("/home/jovyan/dsp/")

# Add code directory to path to import src
# Assuming structure: .../code/data_preparation/extract_network.ipynb
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..'))) 
# Or if running from root, append 'code'
if os.path.exists('code'):
    sys.path.append('code')

from src.data import YearlyTensorDataset

In [36]:
conn = sqlite3.connect('data/patent.db')
cur = conn.cursor()

## Cooccurrence Network

In [37]:
# create edge list for IPC codes by counting co-occurrences for each year of publication
query = """
SELECT 
    ipc1.ipc_code AS ipc1, 
    ipc2.ipc_code AS ipc2, 
    ipc1.YEAR AS pub_year,
    COUNT(*) AS weight
FROM patent_ipc AS ipc1
JOIN patent_ipc AS ipc2 ON ipc1.patent_id = ipc2.patent_id
WHERE ipc1.ipc_code < ipc2.ipc_code
GROUP BY ipc1.ipc_code, ipc2.ipc_code, ipc1.YEAR
"""
df_ipc_cooccurrence = pd.read_sql_query(query, conn)
df_ipc_cooccurrence['pub_year'] = df_ipc_cooccurrence['pub_year'].astype(int)

# Determine the total occurrence (degree) of each IPC code per year to normalize the weights
query_counts = """
SELECT 
    ipc_code, 
    YEAR AS pub_year, 
    COUNT(*) AS count
FROM patent_ipc
GROUP BY ipc_code, YEAR
"""
df_ipc_counts = pd.read_sql_query(query_counts, conn)
df_ipc_counts['pub_year'] = df_ipc_counts['pub_year'].astype(int)

# Merge individual counts back to the edge list
df_ipc_cooccurrence = df_ipc_cooccurrence.merge(
    df_ipc_counts, 
    left_on=['ipc1', 'pub_year'], 
    right_on=['ipc_code', 'pub_year']
).rename(columns={'count': 'count1'}).drop(columns='ipc_code')

df_ipc_cooccurrence = df_ipc_cooccurrence.merge(
    df_ipc_counts, 
    left_on=['ipc2', 'pub_year'], 
    right_on=['ipc_code', 'pub_year']
).rename(columns={'count': 'count2'}).drop(columns='ipc_code')

# Calculate Salton's Cosine Similarity
# Formula: weight / sqrt(count1 * count2)
df_ipc_cooccurrence['salton_similarity'] = df_ipc_cooccurrence['weight'] / np.sqrt(
    df_ipc_cooccurrence['count1'] * df_ipc_cooccurrence['count2']
)

df_ipc_cooccurrence.to_parquet("data/ipc_edgelist_yearly.parquet")

In [7]:
df_ipc_cooccurrence

,ipc1,ipc2,pub_year,weight,count1,count2,salton_similarity
0,a01b1,a01b3,2021,1,5,6,0.182574
1,a01b1,a01b33,2014,1,2,5,0.316228
2,a01b1,a01b5,2021,1,5,2,0.316228
3,a01b1,a01b63,2023,1,7,20,0.084515
4,a01b1,a01b69,2022,1,5,107,0.043234
...,...,...,...,...,...,...,...
891376,h05k7,h05k9,2019,6,281,53,0.049165
891377,h05k7,h05k9,2020,9,306,70,0.061494
891378,h05k7,h05k9,2021,5,298,55,0.039055
891379,h05k7,h05k9,2022,12,381,79,0.069168


## Embedding Similarity Network
Calculate cosine similarity between node embeddings to create edges based on a similarity threshold.
Also calculate the values for the previous years to create edge attributes.

In [100]:
embedding_data = joblib.load("data/embeddings/ipc_mean_year_abstract/ipc_mean_datasets_post2006.pkl")
ipc_code_map = joblib.load("data/ipc_code_map.pkl")
# reverse map
ipc_code_map = {v: k for k, v in ipc_code_map.items()}

In [101]:
ipc_code_map[embedding_data[2006].ipc_code[0].item()]

'a01g7'

In [102]:
# calculate similarity of ipc codes based on embeddings
from sklearn.metrics.pairwise import cosine_similarity
edge_list = []
for year in tqdm(range(2006, 2023)):
    embeddings = embedding_data[year].embeddings.numpy()
    ipc_codes = [ipc_code_map[i.item()] for i in embedding_data[year].ipc_code]

    cosine_sim_matrix = cosine_similarity(embeddings)
    
    for i in range(len(ipc_codes)):
        for j in range(i + 1, len(ipc_codes)):
            sim_value = cosine_sim_matrix[i, j]
            edge_list.append({
                'ipc1': ipc_codes[i],
                'ipc2': ipc_codes[j],
                'pub_year': year,
                'similarity': sim_value
            })


  0%|          | 0/17 [00:00<?, ?it/s]

In [ ]:
df_ipc_similarity = pd.DataFrame(edge_list)

# Sort ipc1 and ipc2 to ensure consistent ordering (alphabetical) for merging
df_ipc_similarity[['ipc1', 'ipc2']] = np.sort(df_ipc_similarity[['ipc1', 'ipc2']], axis=1)
df_ipc_cooccurrence[['ipc1', 'ipc2']] = np.sort(df_ipc_cooccurrence[['ipc1', 'ipc2']], axis=1)

df_merged = df_ipc_similarity.merge(
    df_ipc_cooccurrence,
    on=['ipc1', 'ipc2', 'pub_year'],
    how='outer'
)

In [104]:
df_merged_06 = df_merged[df_merged['pub_year'] == 2006]

In [107]:
df_ipc_cooccurrence_06 = df_ipc_cooccurrence[df_ipc_cooccurrence['pub_year'] == 2006]
df_ipc_similarity_06 = df_ipc_similarity[df_ipc_similarity['pub_year'] == 2006]

In [108]:
df_ipc_cooccurrence_06

,ipc1,ipc2,pub_year,weight,count1,count2,salton_similarity
11,a01b1,a01d34,2006,1,2,30,0.129099
32,a01b1,b25g3,2006,1,2,5,0.316228
49,a01b1,f16d9,2006,1,2,3,0.408248
50,a01b1,f16p3,2006,1,2,4,0.353553
431,a01b45,a01d42,2006,1,6,3,0.235702
...,...,...,...,...,...,...,...
891285,h05k13,h05k7,2006,1,27,69,0.023168
891288,h05k3,h05k5,2006,4,176,48,0.043519
891303,h05k3,h05k7,2006,3,176,69,0.027223
891337,h05k5,h05k7,2006,5,48,69,0.086881


In [106]:
df_merged_06

,ipc1,ipc2,pub_year,similarity,weight,count1,count2,salton_similarity
0,a01g13,a01g7,2006,0.763475,1.0,8.0,10.0,0.111803
1,a01g1,a01g7,2006,0.945883,3.0,9.0,10.0,0.316228
2,a01g7,a01n31,2006,0.682703,1.0,10.0,15.0,0.081650
3,a01g7,c08l33,2006,0.336038,NaN,NaN,NaN,NaN
4,a01g7,c08f20,2006,0.349809,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2297291,c09j153,c23g1,2006,0.289291,NaN,NaN,NaN,NaN
2297292,c09j153,h01f6,2006,0.219100,NaN,NaN,NaN,NaN
2297293,c23g1,g01v11,2006,0.390866,NaN,NaN,NaN,NaN
2297294,g01v11,h01f6,2006,0.333734,NaN,NaN,NaN,NaN


In [109]:
df_merged_06[df_merged_06['similarity'].isna()]

,ipc1,ipc2,pub_year,similarity,weight,count1,count2,salton_similarity
